In [ ]:
from requirements_deprecated import *

from scipy.interpolate import make_interp_spline
from pyscf import scf, dft, gto

# 1. Classical VQE (NumPyMinimumEigensolver)

In [ ]:
def Run_PySCF_HF(dist, basis='sto-3g'):
    try:
        mol = gto.M(atom=f'H 0 0 0; H 0 0 {dist}', basis=basis, verbose=0)
        mf = scf.RHF(mol)
        return mf.kernel()
    except: return None

def Run_PySCF_DFT(dist, basis='sto-3g'):
    try:
        mol = gto.M(atom=f'H 0 0 0; H 0 0 {dist}', basis=basis, verbose=0)
        mf = dft.RKS(mol)
        mf.xc = 'b3lyp'
        return mf.kernel()
    except: return None

def Run_Qiskit_Exact(dist, basis='sto-3g'):
    
    try:
        driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis=basis, charge=0, spin=0)
        problem = driver.run()
        
        hamiltonian_op = problem.hamiltonian.second_q_op()
        
        mapper = JordanWignerMapper()
        qubit_op = mapper.map(hamiltonian_op)
        
        matrix = qubit_op.to_matrix()
        
        eigenvalues = np.linalg.eigvalsh(matrix)
        electronic_energy = eigenvalues[0]
        
        total_energy = electronic_energy + problem.nuclear_repulsion_energy
        
        return total_energy.real
    
    except Exception as e: return None

def Run_Qiskit_VQE(dist, basis='sto-3g'):

    try:
        driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis=basis, charge=0, spin=0)
        problem = driver.run()
        mapper = JordanWignerMapper()
        
        ansatz = UCCSD(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
            initial_state=HartreeFock(
                problem.num_spatial_orbitals,
                problem.num_particles,
                mapper,
            ),
        )
        
        estimator = AerEstimator(
            run_options={"shots": None}, 
            backend_options={"method": "statevector"},
            approximation=True
        )
        
        vqe = VQE(estimator, ansatz, SLSQP())
        vqe.initial_point = [0.0] * ansatz.num_parameters
        
        calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
        result = calc.solve(problem)
        return result.total_energies[0]
    
    except: return None

In [ ]:
distances = np.concatenate([
    np.linspace(0.3, 1.0, 10), 
    np.linspace(1.1, 2.5, 8)
])
distances = np.sort(distances)

results = {
    "Hartree-Fock (Classical)": [],
    "DFT-B3LYP (Classical)": [],
    "Qiskit Exact (FCI)": [],
    "Qiskit VQE (Quantum)": []
}

functions = [
    ("Hartree-Fock (Classical)", Run_PySCF_HF),
    ("DFT-B3LYP (Classical)", Run_PySCF_DFT),
    ("Qiskit Exact (FCI)", Run_Qiskit_Exact),
    ("Qiskit VQE (Quantum)", Run_Qiskit_VQE)
]


for name, func in functions:

    data_points = []

    for d in tqdm(distances):

        energy = func(d)
        data_points.append(energy)

    results[name] = data_points

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['tab:orange', 'tab:green', 'tab:blue', 'tab:red']
styles = ['--', '--', '-', 'o']

for (name, energies), color, style in zip(results.items(), colors, styles):
    
    valid_d = [d for d, e in zip(distances, energies) if e is not None]
    valid_e = [e for e in energies if e is not None]
    
    if not valid_e: continue

    X_smooth = np.linspace(min(valid_d), max(valid_d), 300)
    spline = make_interp_spline(valid_d, valid_e, k=3) 
    Y_smooth = spline(X_smooth)
    
    min_energy = min(Y_smooth)
    min_dist = X_smooth[np.argmin(Y_smooth)]
    
    if "VQE" in name:

        ax.plot(X_smooth, Y_smooth, '-', color=color, alpha=0.5)
        ax.scatter(valid_d, valid_e, color=color, label=f"{name}\nE_min = {min_energy:.4f} Ha", zorder=5)

    else:

        ax.plot(X_smooth, Y_smooth, style, color=color, linewidth=2, label=f"{name}\nE_min = {min_energy:.4f} Ha")
    

    ax.axvline(x=min_dist, linestyle=':', color=color, alpha=0.3)

ax.set_xlabel('Interatomic Distance (Å)', fontsize=12)
ax.set_ylabel('Total Energy (Hartree)', fontsize=12)
ax.set_title('H₂ Ground State', fontsize=14)
ax.grid(True, which='both', linestyle='--', alpha=0.2)
ax.legend(fontsize=10)

plt.show()

In [ ]:
plt.style.use('default') 

plt.rcParams.update({
    'font.size': 14,
    'font.family': 'serif',             
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 13,
    'figure.figsize': (8, 6),           
    'lines.linewidth': 2.5,
    'lines.markersize': 8,
    'axes.linewidth': 1.5,              
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'xtick.direction': 'in',            
    'ytick.direction': 'in',
    'figure.dpi': 300                   
})

fig, ax = plt.subplots()

colors = {
    'Hartree-Fock': '#E69F00',      
    'DFT-B3LYP': '#009E73',         
    'Qiskit Exact': '#0072B2',      
    'Qiskit VQE': '#D55E00'         
}

styles = {
    'Hartree-Fock': '--',
    'DFT-B3LYP': '-.',
    'Qiskit Exact': '-',
    'Qiskit VQE': 'o'              
}

for name, energies in results.items():

    clean_name = name.split(" (")[0]
    
    valid_d = [d for d, e in zip(distances, energies) if e is not None]
    valid_e = [e for e in energies if e is not None]
    
    if not valid_e: continue


    X_smooth = np.linspace(min(valid_d), max(valid_d), 300)
    try:
        spline = make_interp_spline(valid_d, valid_e, k=3)
        Y_smooth = spline(X_smooth)
    except:
        X_smooth, Y_smooth = valid_d, valid_e
    
    min_E = min(Y_smooth)
    
    c = colors.get(clean_name, 'black')
    s = styles.get(clean_name, '-')
    
    if clean_name == "Qiskit VQE":
    
        ax.plot(X_smooth, Y_smooth, '-', color=c, alpha=0.3, linewidth=1.5)
    
        ax.plot(valid_d, valid_e, 'o', color=c, label=f"{clean_name}\n($E_0$ = {min_E:.4f} Ha)", zorder=10)
    else:
        ax.plot(X_smooth, Y_smooth, s, color=c, label=f"{clean_name}\n($E_0$ = {min_E:.4f} Ha)")

ax.set_xlabel(r'Distance $d$ ($\AA$)')
ax.set_ylabel(r'Energy $E$ (Hartree)')
ax.set_title(r'H$_2$ Ground State')

ax.legend(frameon=True, fancybox=False, edgecolor='black', loc='lower right')
ax.set_xlim(0.2, 2.6)
ax.set_ylim(-1.20, -0.95) 

# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

#fig.savefig('h2_groundstate_curve_zoomed.pdf', bbox_inches='tight')

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots()

for name, energies in results.items():
    clean_name = name.split(" (")[0]
    
    valid_d = [d for d, e in zip(distances, energies) if e is not None]
    valid_e = [e for e in energies if e is not None]
    
    if not valid_e: continue

    X_smooth = np.linspace(min(valid_d), max(valid_d), 300)
    try:
        spline = make_interp_spline(valid_d, valid_e, k=3)
        Y_smooth = spline(X_smooth)
    except:
        X_smooth, Y_smooth = valid_d, valid_e
    
    min_E = min(Y_smooth)
    c = colors.get(clean_name, 'black')
    s = styles.get(clean_name, '-')
    
    if clean_name == "Qiskit VQE":
        ax.plot(X_smooth, Y_smooth, '-', color=c, alpha=0.3, linewidth=1.5)
        ax.plot(valid_d, valid_e, 'o', color=c, label=f"{clean_name}\n($E_0$ = {min_E:.4f} Ha)", zorder=10)
    else:
        ax.plot(X_smooth, Y_smooth, s, color=c, label=f"{clean_name}\n($E_0$ = {min_E:.4f} Ha)")


ax.set_xlabel(r'Distance $d$ ($\AA$)')
ax.set_ylabel(r'Energy $E$ (Hartree)')
ax.set_title(r'H$_2$ Ground State')

ax.set_ylim(-1.20, -0.4) 
ax.set_xlim(0.2, 2.6)

ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=6))

ax.legend(frameon=True, fancybox=False, edgecolor='black', loc='upper right')

plt.tight_layout()
plt.show()

fig.savefig('h2_groundstate_curve.pdf', bbox_inches='tight')

# 2. Quantum VQE (Qiskit Estimator V1 & Qiskit Aer Estimator - Primitive V1)

In [ ]:
def Run_VQE_Specific_Estimator(dist, estimator_type="statevector", shots=None, seed=42):
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    if estimator_type == "statevector":
        estimator = AerEstimator(
            run_options={"shots": None},
            transpile_options={"seed_transpiler": seed},
            approximation=True
        )
        
    elif estimator_type == "sampling":
        estimator = AerEstimator(
            run_options={"shots": shots, "seed": seed},
            transpile_options={"seed_transpiler": seed},
            approximation=False
        )
        
    elif estimator_type == "backend":

        sim_backend = AerSimulator()
        estimator = BackendEstimator(backend=sim_backend)
        estimator.set_options(shots=shots, seed=seed)

    if estimator_type == "statevector":
        
        optimizer = SLSQP(maxiter=100)
    
    else:

        from qiskit_algorithms.optimizers import COBYLA
        optimizer = COBYLA(maxiter=100)

    vqe = VQE(estimator, ansatz, optimizer)
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    return result.total_energies[0]

In [ ]:
distances = np.concatenate([np.linspace(0.3, 0.9, 8), np.linspace(1.0, 2.5, 6)])
distances = np.sort(distances)

results_est = {
    "Aer (Statevector)": [],
    "Aer (1024 Shots)": [],
    "BackendEstimator (Sim)": []
}

print("Starting Estimator Comparison...")

for d in tqdm(distances):
    try:
        e1 = Run_VQE_Specific_Estimator(d, "statevector", shots=None)
        results_est["Aer (Statevector)"].append(e1)
    except: results_est["Aer (Statevector)"].append(None)
    
    try:
        e2 = Run_VQE_Specific_Estimator(d, "sampling", shots=1024)
        results_est["Aer (1024 Shots)"].append(e2)
    except: results_est["Aer (1024 Shots)"].append(None)
    
    try:
        e3 = Run_VQE_Specific_Estimator(d, "backend", shots=1024)
        results_est["BackendEstimator (Sim)"].append(e3)
    except: results_est["BackendEstimator (Sim)"].append(None)

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(8, 6))

styles_est = {
    "Aer (Statevector)": {"c": "#0072B2", "fmt": "-", "lw": 2.5, "alpha": 1.0, "z": 1},   # Solid Blue
    "Aer (1024 Shots)":  {"c": "#D55E00", "fmt": "o", "lw": 0, "alpha": 0.8, "z": 2},     # Orange Dots
    "BackendEstimator (Sim)": {"c": "#009E73", "fmt": "x", "lw": 0, "alpha": 0.8, "z": 3} # Green Crosses
}

for name, energies in results_est.items():
    valid_d = [d for d, e in zip(distances, energies) if e is not None]
    valid_e = [e for e in energies if e is not None]
    
    if not valid_e: continue
    
    s = styles_est[name]
    
    if "Statevector" in name:
        X_smooth = np.linspace(min(valid_d), max(valid_d), 300)
        try:
            spline = make_interp_spline(valid_d, valid_e, k=3)
            Y_smooth = spline(X_smooth)
            ax.plot(X_smooth, Y_smooth, color=s["c"], linewidth=s["lw"], label=name, zorder=s["z"])
        except:
            ax.plot(valid_d, valid_e, color=s["c"], linewidth=s["lw"], label=name, zorder=s["z"])

    else:
        ax.scatter(valid_d, valid_e, marker=s["fmt"], color=s["c"], alpha=s["alpha"], label=name, zorder=s["z"])

ax.set_xlabel(r'Distance $d$ ($\AA$)')
ax.set_ylabel(r'Energy $E$ (Hartree)')
ax.set_title(r'Comparison of VQE Estimators (H$_2$)')

ax.set_ylim(-1.20, -0.4) 
ax.set_xlim(0.2, 2.6)

ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=6))

ax.legend(frameon=True, fancybox=False, edgecolor='black', loc='lower right')
plt.tight_layout()
plt.show()

# 3. 